# Day 23: Bridge to LLMs

We trained small models ourselves. Now we look **inside** models that are far too big for a laptop (ChatGPT and friends).

We follow **one sentence** the whole day:

> **"ali drinks hot ___"** → the model guesses **chai**

**The map** (same as the slides):

| Text | → Tokens | → IDs | → Embed | → Attention | → Scores | → Softmax | → Next |
|---|---|---|---|---|---|---|---|
| ali drinks hot | 3 tokens | 0 1 3 | 3 vectors | 1 mixed vector | 6 scores | 6 chances | "chai" |

### New words today

| Word | Meaning |
|---|---|
| **LLM** | Large Language Model. A huge network that guesses the next token. |
| **Token** | A piece of text: a word or part of a word. |
| **Vocabulary** | The list of all tokens the model knows. |
| **Token ID** | The position of a token in the vocabulary. |
| **Embedding** | A vector (list of numbers) for each token. |
| **Dot product** | Multiply matching numbers, then add. Big = similar. |
| **Attention** | A weighted average: each word looks back at the other words. |
| **Next-token prediction** | Give every token a chance, pick one, repeat. |

In [ ]:
import numpy as np                      # vectors and tables of numbers
np.set_printoptions(precision=3, suppress=True)   # print 3 decimals, no e-notation


def softmax(scores):                    # same softmax as Day 21
    e = np.exp(scores)                  # 1. e to the power of each score
    return e / e.sum()                  # 2. divide by the total → chances that add up to 1

---
# Class Problem: "ali drinks hot ___"

## Step 1: Tokens

- A computer cannot read words. First we **cut the text into tokens**.
- Our toy: **1 word = 1 token** (cut at the spaces).

In [ ]:
sentence = "ali drinks hot"
tokens = sentence.split()        # cut at the spaces
print(tokens)                    # ['ali', 'drinks', 'hot']

**A real tokenizer** (the one used by GPT-4) cuts words into **pieces**.
If this cell gives an error, run `pip install tiktoken` once (it needs internet the first time).

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")          # the GPT-4 tokenizer
for text in ["Ali drinks hot chai.", "chaiwala", "unbelievable", "Lahore"]:
    ids = enc.encode(text)                           # text → token IDs
    pieces = [enc.decode([i]) for i in ids]          # each ID → its piece of text
    print(f"{text!r:24} → {pieces}  ({len(ids)} tokens)")

**✏️ Quick Check:** Use `enc.encode` to count the tokens in **your own name** and in **"Karachi"**.

In [ ]:
# your code here

**Answer:**

In [ ]:
for text in ["Sara", "Karachi"]:
    ids = enc.encode(text)
    print(text, "→", [enc.decode([i]) for i in ids], len(ids), "tokens")

## Step 2: Token IDs

- The **vocabulary** is the list of all tokens the model knows.
- The **ID** of a token = its position in that list.
- Our vocabulary has 6 words. A real one has about 100,000.

In [ ]:
vocab = ["ali", "drinks", "eats", "hot", "chai", "roti"]   # position 0, 1, 2, 3, 4, 5

ids = [vocab.index(t) for t in tokens]    # find each token's position
print(ids)                                # [0, 1, 3]

**✏️ Quick Check:** What are the IDs of **"ali eats hot"**? Guess first, then check with code.

In [ ]:
# your code here

**Answer:**

In [ ]:
print([vocab.index(t) for t in "ali eats hot".split()])   # [0, 2, 3]

**Think:** what happens with `vocab.index("sara")`? Try it. Real tokenizers never get this error. Why? *(They cut unknown words into smaller pieces they do know, like "chaiwala" → chai + w + ala.)*

## Step 3: Embeddings (vectors)

- Each token becomes a **vector**: a list of numbers.
- Our vectors have 4 numbers with names: **person, drink, food, warm**.
- Real models: 768 or more numbers per token, **learned**, with no names. We wrote ours to understand.

In [ ]:
#                person drink food warm
E = np.array([[3,     0,    0,   0],    # 0 ali
              [0,     2,    0,   0],    # 1 drinks
              [0,     0,    2,   0],    # 2 eats
              [0,     1,    1,   1],    # 3 hot   (drink? food? not sure!)
              [0,     2,    0,   1],    # 4 chai
              [0,     0,    2,   1]],   # 5 roti
             dtype=float)

vectors = E[ids]           # take the rows for our IDs [0, 1, 3]
print(vectors)

**✏️ Quick Check:** Print the vector of **roti** using its ID.

In [ ]:
# your code here

**Answer:**

In [ ]:
print(E[vocab.index("roti")])     # [0, 0, 2, 1]: food, warm

## Step 4: Dot product = similarity

- **Multiply the matching numbers, then add.**
- Big number = similar meaning. 0 = nothing in common.
- Same "multiply and add" as a CNN filter and a Dense neuron.

In [ ]:
chai = E[vocab.index("chai")]
for word in vocab:
    print(f"chai · {word:7} = {np.dot(chai, E[vocab.index(word)])}")

**✏️ Quick Check:** Which word is **most similar to roti** (not counting roti itself)? Use `np.dot`.

In [ ]:
# your code here

**Answer:**

In [ ]:
roti = E[vocab.index("roti")]
for word in vocab:
    print(f"roti · {word:7} = {np.dot(roti, E[vocab.index(word)])}")
# eats: 4  → roti is most similar to "eats"

## Step 5: The problem: "hot" alone cannot decide

Use only the last word, "hot", to score every word in the vocabulary:

In [ ]:
hot = E[vocab.index("hot")]
for word, score in zip(vocab, E @ hot):     # E @ hot = dot product of hot with every row
    print(f"{word:7} {score}")
# chai 3, roti 3, hot 3 → a tie! "hot" needs to look back at "drinks".

## Step 6: Attention

1. **Score:** dot product of "hot" with each word in the sentence.
2. **Softmax:** scores → weights that add up to 1.
3. **Weighted average:** multiply each word's vector by its weight, then add.

In [ ]:
query = vectors[-1]                  # the last word: "hot"

att_scores = vectors @ query         # 1. hot · ali, hot · drinks, hot · hot
print("scores: ", att_scores)        # [0, 2, 3]

weights = softmax(att_scores)        # 2. softmax
print("weights:", weights)           # [0.035, 0.259, 0.705]

mix = weights @ vectors              # 3. weighted average of the 3 vectors
print("hot before:", query)          # [0, 1, 1, 1]
print("hot after: ", mix)            # [0.105, 1.224, 0.705, 0.705]: drink went up!

**✏️ Quick Check:** Do the weights add up to 1? Print `weights.sum()`. Which word does "hot" listen to the least?

In [ ]:
# your code here

**Answer:**

In [ ]:
print(round(weights.sum(), 3))     # 1.0 (the computer may show 0.9999999: tiny rounding)
# "ali": only 3.5%. A name tells us nothing about chai or roti.

## Step 7: Score every word, softmax, pick the next token

In [ ]:
scores = E @ mix                     # dot product of the mixed "hot" with every word
chances = softmax(scores)

for i in np.argsort(-chances):       # biggest chance first
    print(f"{vocab[i]:7} score {scores[i]:.2f}   chance {chances[i]:.1%}")

print("Next token:", vocab[chances.argmax()])    # chai

**Why is "hot" in 2nd place?** Our toy is tiny and uses the same vectors to read and to guess. A real model has a separate, learned output layer and has read enough text to know that "hot hot" is rare.

## Step 8: Put it together, and repeat

One function = the whole map. Then we **add the new token and run again**.

In [ ]:
def next_token(words):
    vectors = E[[vocab.index(w) for w in words]]      # tokens → IDs → vectors
    weights = softmax(vectors @ vectors[-1])          # attention weights for the last word
    mix = weights @ vectors                           # weighted average
    chances = softmax(E @ mix)                        # score every word → chances
    return vocab[chances.argmax()], chances


words = ["ali", "drinks", "hot"]
for step in range(3):                                 # 3 passes = 3 new tokens
    word, chances = next_token(words)
    print(" ".join(words), "→", word, f"({chances.max():.0%})")
    words.append(word)                                # add the guess, then repeat

**✏️ Quick Check:** The toy says "chai chai chai". Why would a real LLM not do this?

**Answer:** Our toy has only 6 words and was never trained. A real LLM learned from huge amounts of text that "chai chai chai" almost never happens, so it gives other tokens a bigger chance. (Real chatbots also sometimes pick a token that is not the top one.)

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [ ]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

# 1–2. tokens → IDs
vocab = ["ali", "drinks", "eats", "hot", "chai", "roti"]
tokens = "ali drinks hot".split()
ids = [vocab.index(t) for t in tokens]

# 3. embeddings: person, drink, food, warm
E = np.array([[3, 0, 0, 0], [0, 2, 0, 0], [0, 0, 2, 0],
              [0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 1]], dtype=float)
vectors = E[ids]

# 4–6. attention for the last word
weights = softmax(vectors @ vectors[-1])
mix = weights @ vectors

# 7. score every word, softmax, pick
chances = softmax(E @ mix)
print("Next token:", vocab[chances.argmax()], f"({chances.max():.1%})")   # chai (37.4%)

### Key takeaways

- An LLM has **one job**: guess the next token, then repeat.
- Text → **tokens** → **IDs** → **vectors** (embeddings).
- **Dot product** = similarity. **Attention** = a weighted average, so each word can use the words before it.
- **Softmax** turns scores into chances (same as Day 21). Training uses **cross entropy** (same as Day 22).
- Real LLMs do the same steps with learned numbers, many layers, and billions of weights.

---
# 🏠 Home Problem: "ali eats hot ___"

Do the **same steps** for the new sentence.

1. Tokens and IDs.
2. Vectors (`E[ids]`).
3. Attention: scores, weights, mix. Compare "hot" before and after: which number went up?
4. Score every word, softmax. What is the next token, and its chance?
5. In 2 lines: only one word changed. Which step made the answer change, and how?